# Check an S3 bucket

Lists what's in a bucket on EEA's S3-compatible store (`http://sggw.pdmz.eea:10445`).

Credentials are never written in this notebook. They're read from the kernel environment, or else from the first `.env` found in this folder or a parent (git-ignored):

```
S3_ACCESS_KEY_ID=...
S3_SECRET_ACCESS_KEY=...
S3_ENDPOINT_URL=http://sggw.pdmz.eea:10445   # optional, this is the default
S3_BUCKET=testbucket                          # optional, this is the default
S3_REGION=us-east-1                           # optional, this is the default
S3_VERIFY_SSL=true                            # optional — see below
```

**HTTPS and `S3_VERIFY_SSL`:** the gateway's HTTPS port (`:10446`) presents a valid certificate, but it is issued for `*.eea.europa.eu`, while the gateway is only reachable as `sggw.pdmz.eea`. Verification therefore fails with a hostname mismatch. `S3_VERIFY_SSL=false` skips the check: traffic is still encrypted, but the server's identity isn't verified. The plain-HTTP port (`:10445`) needs no certificate, but sends data unencrypted (the keys themselves are never sent, only signatures).

## Settings

In [ ]:
import os

import boto3
import pandas as pd
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError

from eea_datalakehouse.session_utils import find_dotenv, read_dotenv_value

DEFAULTS = {
    "S3_ENDPOINT_URL": "https://s3datahub.eea.europa.eu:8080",
    "S3_BUCKET": "dh-prod-data",
    "S3_REGION": "us-east-1",
    "S3_VERIFY_SSL": "True",
    "S3_ACCESS_KEY_ID" : "CCWNR392OGHHXKLVXVU3",
    "S3_SECRET_ACCESS_KEY" : "_OAppJ_c8VxkW932_Htfxy_t6r_74pBZQzPhA1Bk",
}
DOTENV = find_dotenv()


def setting(name: str) -> str | None:
    """`name` from the environment, else from the .env, else its default."""
    value = os.environ.get(name) or (read_dotenv_value(DOTENV, name) if DOTENV else None)
    return value or DEFAULTS.get(name)


ENDPOINT = DEFAULTS["S3_ENDPOINT_URL"]
BUCKET = DEFAULTS["S3_BUCKET"]  
ACCESS_KEY = DEFAULTS["S3_ACCESS_KEY_ID"]
SECRET_KEY = DEFAULTS["S3_SECRET_ACCESS_KEY"]
VERIFY_SSL = DEFAULTS["S3_VERIFY_SSL"].strip().lower() not in ("false", "0", "no")

print(f"endpoint: {ENDPOINT}")
print(f"bucket:   {BUCKET}")
if ENDPOINT.startswith("https"):
    print(f"verify:   {'certificate checked' if VERIFY_SSL else 'OFF — server identity not verified'}")
print(f".env:     {DOTENV or 'none found'}")
print(f"keys:     {'set' if ACCESS_KEY and SECRET_KEY else 'MISSING — add them to .env'}")

## Client

Path-style addressing (`endpoint/bucket/key`) is forced: an on-prem gateway like this one doesn't resolve `bucket.host` virtual-host names.

In [ ]:
import urllib3

if not VERIFY_SSL:
    # Asked for explicitly via S3_VERIFY_SSL=false — don't repeat the warning on every request.
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    region_name=setting("S3_REGION"),
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    verify=True,
    config=Config(
        s3={"addressing_style": "path"},
        signature_version="s3v4",
        connect_timeout=10,
        read_timeout=30,
        retries={"max_attempts": 2},
    ),
)

## List the bucket

`PREFIX` limits the listing to keys under it (`""` = whole bucket); `LIMIT` stops after that many keys (`None` = all, every page is read).

In [ ]:
PREFIX = ""
LIMIT = None


def list_objects(prefix: str = "", limit: int | None = None) -> pd.DataFrame:
    rows = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            rows.append(
                {"key": obj["Key"], "size": obj["Size"], "last_modified": obj["LastModified"]}
            )
            if limit is not None and len(rows) >= limit:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows, columns=["key", "size", "last_modified"])


try:
    objects = list_objects(PREFIX, LIMIT)
except ClientError as exc:
    error = exc.response.get("Error", {})
    status = exc.response.get("ResponseMetadata", {}).get("HTTPStatusCode")
    raise SystemExit(f"S3 refused the listing ({status} {error.get('Code')}): {error.get('Message')}")
except BotoCoreError as exc:
    hint = (
        " — the certificate doesn't match the host name; see S3_VERIFY_SSL at the top"
        if "SSL" in str(exc)
        else ""
    )
    raise SystemExit(f"could not reach {ENDPOINT}: {exc}{hint}")

print(f"s3://{BUCKET}/{PREFIX}: {len(objects)} object(s), {objects['size'].sum() / 1024**2:.1f} MB")
objects

## Summary by top-level folder

In [ ]:
if objects.empty:
    print("(empty)")
else:
    folders = objects.assign(folder=objects["key"].str.split("/", n=1).str[0])
    summary = folders.groupby("folder").agg(
        objects=("key", "size"), size_mb=("size", lambda s: round(s.sum() / 1024**2, 1)),
        newest=("last_modified", "max"),
    )
    display(summary.sort_values("size_mb", ascending=False))